The goal for this file is twofold: 
1) To tag each municipality as 1st to 5th income class as provided by the DOF
2) To the connect the Immunization data from 2018 to 2026 as provided by the FSIS.

In [7]:
import pandas as pd
from functools import reduce

# Tagging the Income Classification

With the Excel file provided about the PSGC codes, the tags on the income class are not actually provided. [DeepSeek](https://chat.deepseek.com/share/34sb7ym87whiiwjsj3) was used to convert [DOF PDF](https://blgf.gov.ph/wp-content/uploads/2025/01/DOF-DO-074.2024.pdf) into an Excel. Further cleaning will be done because it only has the names of the municipalities, and not the actual codes.

In [8]:
income_df = pd.read_excel('data/income/income.xlsx').drop_duplicates()
income_df

,REGION,Province Name,Geographic Level,LGU NAME,OLD CLASS,NEW CLASS
0,CAR,Abra,Province,Abra,3rd,1st
1,CAR,Apayao,Province,Apayao,3rd,2nd
2,CAR,Benguet,Province,Benguet,2nd,1st
3,CAR,Ifugao,Province,Ifugao,3rd,2nd
4,CAR,Kalinga,Province,Kalinga,3rd,2nd
...,...,...,...,...,...,...
1824,BARMM,Tawi-Tawi,Mun,Simunul,4th,2nd
1825,BARMM,Tawi-Tawi,Mun,Sitangkai,1st,1st
1826,BARMM,Tawi-Tawi,Mun,South Ubian,3rd,2nd
1827,BARMM,Tawi-Tawi,Mun,Tandubas,2nd,1st


# Adding the KPIs from the FHSIS

From a first look through file system and the files within it, we know that:
* The granularity of the data vary based on the years. We have that:
    * For 2018 to 2024 data, it is annual, and we only have at most provincial- and city- level data.
    * For 2025 and 2026 data, the granularity is monthly and municipal-level. However, the 2026 data is only until February.
* The key performance indicators that the FHSIS reports are:
    * Fully Immunized Children (FIC) - The monthly target is 7.92%, which comes from the annual target of 95% divided by the 12 months.
        * According to the [DOH](https://doh.gov.ph/wp-content/uploads/2023/08/Booklet-7-Monitoring-Supportive-Supervision-and-Evaluation.pdf), for a child to be fully immunized, they need to have the following in the list. The FSIS also report the vaccination rates below:
            * One (1) dose of bCG at birth or anytime, 
            * Three (3) doses of OPV, 
            * Three (3) doses of Pentavalent vaccines; and 
            * One (1) dose of Measles-containing vaccine (MCV)
                * For this one, the FSIS records this as Two (2) doses of Measles Mumps Rubella (2 MMR) vaccine
    * Completely Immunized Children (CIC) - This one does not seem as important

Overall plan:
* Because of the differing granularities of the dataset, we will have two separate dataframes. They are:
    * Annual and provincial-level data from 2018 to 2025
    * Monthly and municipal-level data from 2025 to February 2026

For the naming schemes they are:

1. Annual Immunization Dataset – Column Definitions

| Column Name  | Description                                          |
| ------------ | ---------------------------------------------------- |
| Area         | Geographic area (e.g., region, province, or city)    |
| Eligible_Pop | Total number of individuals eligible for vaccination |

2. At Birth Indicators

| Column Name | Description                                                                                  |
| ----------- | -------------------------------------------------------------------------------------------- |
| CPAB        | Children Protected at Birth; infants protected against tetanus through maternal immunization |
| BCG         | Bacillus Calmette–Guérin vaccine; protects against tuberculosis                              |
| HEPA_B1     | Hepatitis B Birth Dose; first dose given at birth                                            |

3. Primary Series Vaccines

| Column Name | Description                                                             |
| ----------- | ----------------------------------------------------------------------- |
| DPT_1       | First dose of DPT-containing vaccine (Diphtheria, Pertussis, Tetanus)   |
| DPT_2       | Second dose of DPT-containing vaccine                                   |
| DPT_3       | Third dose of DPT-containing vaccine (completion of primary DPT series) |
| OPV_1       | First dose of Oral Polio Vaccine                                        |
| OPV_2       | Second dose of Oral Polio Vaccine                                       |
| OPV_3       | Third dose of Oral Polio Vaccine                                        |
| IPV         | Inactivated Polio Vaccine                                               |

4. Pneumococcal Vaccines

| Column Name | Description                                   |
| ----------- | --------------------------------------------- |
| PCV_1       | First dose of Pneumococcal Conjugate Vaccine  |
| PCV_2       | Second dose of Pneumococcal Conjugate Vaccine |
| PCV_3       | Third dose of Pneumococcal Conjugate Vaccine  |

5. Measles-Containing Vaccines

| Column Name | Description                               |
| ----------- | ----------------------------------------- |
| MCV_1       | First dose of Measles-Containing Vaccine  |
| MCV_2       | Second dose of Measles-Containing Vaccine |

6. Immunization Coverage Indicators

| Column Name | Description                                                                                                      |
| ----------- | ---------------------------------------------------------------------------------------------------------------- |
| FIC         | Fully Immunized Child; child who has received all recommended basic vaccines                                     |
| CIC         | Completely Immunized Child; broader definition depending on program (may include additional vaccines beyond FIC) |

---

Each vaccination indicator is further disaggregated and represented using the following suffix-based naming convention:

* `_M` for Male recipients
* `_F` for Female recipients
* `_Total` for Total recipients (Male + Female)
* `_Percent` for Proportion of vaccinated individuals relative to the eligible population

This means that for each base indicator (e.g., `DPT_1`, `BCG`, `FIC`), the dataset contains four corresponding columns:

* `{Indicator}_M`
* `{Indicator}_F`
* `{Indicator}_Total`
* `{Indicator}_Percent`

### Cleaning 2018 Data

Based on the table summary, we need the following tables:
* Table 2D.1	Proportion of FIC, CIC and Children Protected at Birth
* Table 2D.2	Proportion of Children given BCG and Hepatitis B1 Vaccines
* Table 2D.3	Proportion of Children given Pentavalent Vaccines
* Table 2D.4	Proportion of Children given Oral Polio Vaccines (OPV)
* Table 2D.5	Proportion of Children given Measles-containing Vaccines (MCV) and Rotavirus Vaccines


In [9]:
TEMP_COLUMN_NAMES = [
 'Area',
 'Eligible_Pop',
 'CPAB',
 'BCG',
 'HEPA_B1',
 'DPT_1',
 'DPT_2',
 'DPT_3',
 'OPV_1',
 'OPV_2',
 'OPV_3',
 'IPV',
 'PCV_1',
 'PCV_2',
 'PCV_3',
 'MCV_1',
 'MCV_2',
 'FIC',
 'CIC']

CODE_COLUMN_NAMES = TEMP_COLUMN_NAMES[:2]
for indicator in TEMP_COLUMN_NAMES[2:]:
    CODE_COLUMN_NAMES.append(indicator + '_M')
    CODE_COLUMN_NAMES.append(indicator + '_F')
    CODE_COLUMN_NAMES.append(indicator + '_Total')
    CODE_COLUMN_NAMES.append(indicator + '_Percent')

def create_column_names(original_column_names):
    """Add the _M, _F, _Total, _Percent to all indicators"""
    code_column_names = ['Area', 'Eligible_Pop']

    for indicator in original_column_names:
        code_column_names.append(indicator + '_M')
        code_column_names.append(indicator + '_F')
        code_column_names.append(indicator + '_Total')
        code_column_names.append(indicator + '_Percent')

    return code_column_names

CODE_COLUMN_NAMES.insert(2, 'Year')

In [10]:
# 2018 data

# TABLE 1
table_1_2018_df = pd.read_excel('data/medical/2018-2023/CC 2018.xlsx', 
                                   sheet_name='Table 2D.1',
                                   skiprows=7,
                                   skipfooter=3)

# get rid of the 10th column (number of live births) and the last column
table_1_2018_df = table_1_2018_df.drop(table_1_2018_df.columns[10], axis=1).copy()
table_1_2018_df = table_1_2018_df.drop(table_1_2018_df.columns[-1], axis=1).copy()

# rename columns
table_1_2018_df.columns = create_column_names(['FIC', 'CIC', 'CPAB'])
table_1_2018_df.dropna(inplace=True)

# TABLE 2
table_2_2018_df = pd.read_excel('data/medical/2018-2023/CC 2018.xlsx', 
                                sheet_name='Table 2D.2',
                                skiprows=7,
                                skipfooter=3)

# drop the last three columns
table_2_2018_df = table_2_2018_df.drop(table_2_2018_df.columns[-4:], axis=1).copy()
table_2_2018_df.columns = create_column_names(['BCG', 'HEPA_B1'])
table_2_2018_df.dropna(inplace=True)

# TABLE 3
table_3_2018_df = pd.read_excel('data/medical/2018-2023/CC 2018.xlsx', 
                                sheet_name='Table 2D.3',
                                skiprows=7,
                                skipfooter=3)

table_3_2018_df.columns = create_column_names(['DPT_1', 'DPT_2', 'DPT_3'])
table_3_2018_df.dropna(inplace=True)

# TABLE 4 
table_4_2018_df = pd.read_excel('data/medical/2018-2023/CC 2018.xlsx', 
                                sheet_name='Table 2D.4',
                                skiprows=7,
                                skipfooter=3)

table_4_2018_df.columns = create_column_names(['OPV_1', 'OPV_2', 'OPV_3'])
ipv_cols = create_column_names(['IPV'])[2:]
for ipv_col in ipv_cols:
    table_4_2018_df[ipv_col] = 0
table_4_2018_df.dropna(inplace=True)

# TABLE 5
table_5_2018_df = pd.read_excel('data/medical/2018-2023/CC 2018.xlsx', 
                                sheet_name='Table 2D.5',
                                skiprows=7,
                                skipfooter=3)
table_5_2018_df = table_5_2018_df.drop(table_5_2018_df.columns[-8:], axis=1).copy()
table_5_2018_df.columns = create_column_names(['MCV_1', 'MCV_2'])
table_5_2018_df.dropna(inplace=True)

# TABLE 6
table_6_2018_df = pd.read_excel('data/medical/2018-2023/CC 2018.xlsx', 
                                sheet_name='Table 2D.6',
                                skiprows=7,
                                skipfooter=3)
table_6_2018_df.columns = create_column_names(['PCV_1', 'PCV_2', 'PCV_3'])
table_6_2018_df['Area'] = table_6_2018_df['Area'].replace('N C R 1', 'N C R')
table_6_2018_df.dropna(inplace=True)

In [27]:
table_dfs = [
    table_1_2018_df,
    table_2_2018_df,
    table_3_2018_df,
    table_4_2018_df,
    table_5_2018_df,
    table_6_2018_df
]

table_2018_df = pd.concat(table_dfs, axis=1, join='inner')
table_2018_df

,Area,Eligible_Pop,FIC_M,FIC_F,FIC_Total,FIC_Percent,CIC_M,CIC_F,CIC_Total,CIC_Percent,...,PCV_1_Total,PCV_1_Percent,PCV_2_M,PCV_2_F,PCV_2_Total,PCV_2_Percent,PCV_3_M,PCV_3_F,PCV_3_Total,PCV_3_Percent
0,PHILIPPINES,2866557.681,977139.0,919976.0,1897115.0,66.180946,110441.0,102822.0,213263.0,7.439690,...,1255346.0,43.792804,645422.624051,609927.248102,1.255350e+06,43.792940,627843.0,598075.4,1225918.4,42.766221
2,N C R,363987.972,135397.0,130973.0,266370.0,73.180990,14557.0,13689.0,28246.0,7.760147,...,76.0,0.020880,0.000000,0.000000,0.000000e+00,0.000000,1.0,0.0,1.0,0.000275
4,Malabon,10331.928,3183.0,3032.0,6215.0,60.153342,2797.0,2725.0,5522.0,53.445978,...,0.0,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.000000
5,Navotas,7051.320,2846.0,2800.0,5646.0,80.070115,408.0,445.0,853.0,12.097026,...,0.0,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.000000
6,Valenzuela City,17536.824,6996.0,6665.0,13661.0,77.898940,970.0,916.0,1886.0,10.754513,...,0.0,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244,Surigao del Sur,13845.384,5176.0,4940.0,10116.0,73.064062,187.0,152.0,339.0,2.448469,...,10142.0,73.251851,5148.000000,4957.000000,1.010500e+04,72.984614,5050.0,4852.0,9902.0,71.518421
245,Province of Dinagat,3438.342,1087.0,983.0,2070.0,60.203435,106.0,120.0,226.0,6.572935,...,1813.0,52.728902,1034.000000,959.000000,1.993000e+03,57.963984,1048.0,932.0,1980.0,57.585895
247,Bislig City,2629.773,1008.0,993.0,2001.0,76.090218,13.0,7.0,20.0,0.760522,...,1877.0,71.374982,962.000000,886.000000,1.848000e+03,70.272225,898.0,849.0,1747.0,66.431589
248,Butuan City,9482.265,4053.0,3715.0,7768.0,81.921355,123.0,139.0,262.0,2.763053,...,6767.0,71.364806,3364.000000,3220.000000,6.584000e+03,69.434887,3315.0,3096.0,6411.0,67.610429
